# IOAI — 2024 Selection Test Cv Cifar Resnet (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
print('CIFAR-10 은 노트북이 torchvision 으로 자동 다운로드합니다.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# CIFAR-10 Image Classification via ResNet

> **Singapore IOAI 2024 — Selection Test (CV)**

Image classification on CIFAR-10 with a **ResNet** (here ResNet18, ImageNet-pretrained). Score = **test accuracy**. The notebook downloads CIFAR-10, trains, and predicts the test set in canonical order; **Submit** scores `submission.csv` (`id,label`) against the test labels.

In [ ]:
import numpy as np, pandas as pd, torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

## Data — CIFAR-10 (downloaded). Test order is canonical (shuffle=False) to match the grading key.

In [ ]:
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
tf_train = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
                               transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
tf_test = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
train_ds = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=tf_train)
test_ds = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=tf_test)
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=2)
test_dl = DataLoader(test_ds, batch_size=512, shuffle=False, num_workers=2)
len(train_ds), len(test_ds)

## Model — ResNet18 (ImageNet-pretrained) fine-tuned to 10 classes

In [ ]:
def build_model():
    m = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, 10)
    return m.to(device)

model = build_model()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate(dl):
    model.eval(); correct = total = 0
    for x, y in dl:
        p = model(x.to(device)).argmax(1).cpu()
        correct += (p == y).sum().item(); total += y.size(0)
    return correct / total

for epoch in range(3):
    model.train()
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(); crit(model(x), y).backward(); opt.step()
    print(f'epoch {epoch+1}  test_acc {evaluate(test_dl):.4f}')

## Predict the test set (canonical order) → `submission.csv` (`id,label`)

In [ ]:
model.eval(); preds = []
with torch.no_grad():
    for x, _ in test_dl:
        preds.append(model(x.to(device)).argmax(1).cpu().numpy())
preds = np.concatenate(preds)
pd.DataFrame({'id': range(len(preds)), 'label': preds}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(preds))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)